# Task 1 — Universal Multi-Corruption Denoising Autoencoder (Colab runner)

One autoencoder restores clean, salt-and-pepper, blurred and occluded pet images without being told which corruption was applied. Run the cells **from top to bottom** on a **T4 GPU** (Runtime → Change runtime type → T4 GPU).

**If Colab disconnects:** reconnect, run cells 1 and 2 again, then re-run the cell that was interrupted. Everything is saved to Google Drive and resumes where it stopped: the Optuna study continues with the remaining trials, training continues from the last finished epoch (`--resume`), and cells that already finished return immediately (training) or simply recompute their outputs (evaluation, export).

**Runtime (ESTIMATES, not measured on a T4):** about 2–2.5 h in total — Optuna ≈ 1.5 h (capped at 90 min per session), final training ≈ 10–20 min, evaluation ≈ 5 min, ONNX export ≈ 1 min, ablation ≈ 30–50 min. The real time per epoch is printed in every training log line; use it to adjust `--epochs` / `--n-trials` if needed. Design decisions and all commands: `docs/task1_notes.md`.

## 1. Get the code

In [ ]:
# 1. code
REPO_URL = 'https://github.com/fsdev87/generative-ai-A1.git'
import os
if os.path.exists('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone {REPO_URL} /content/repo
%cd /content/repo

## 2. Setup: Drive, packages, data cache, W&B, environment variables
Needs the Colab secret `WANDB_API_KEY`. Estimated time: 1–3 min (first session longer: downloads and caches the dataset).

In [ ]:
# 2. setup: Drive, packages, data cache, W&B, env vars
from src.common.colab import setup
setup(fs2k=False)   # True for Task 4

## 3. Quick check (optional, ≈ 1 min)
A tiny run on synthetic data (W&B disabled, written to a separate `smoke` folder, so it never touches real results). If this fails, fix the problem before spending GPU time.

In [ ]:
!python -m src.task1.train --smoke

## 4. Optuna hyperparameter search (estimate ≈ 1.5 h)
Searches learning rate, batch size, bottleneck size (`latent_channels`), encoder width (`base_channels`), dropout and the loss weight α. Each trial trains 15 epochs (a quarter of the final run); weak trials are stopped early by the median pruner; every trial is a W&B run in group `task1-optuna`. Trial 0 is the brief's starting point (α = 0.8) as a reference.

The study stops after 40 finished trials or when no new trial may start after 90 minutes. **After a disconnect just run this cell again** — finished trials are kept on Drive and only the remaining ones run. Results: `OUTPUT_DIR/task1/optuna/task1_udae/` (plots, `trials.csv`) and the selected configuration `OUTPUT_DIR/task1/best_config.yaml`.

In [ ]:
!python -m src.task1.optuna_search --n-trials 40 --epochs 15 --timeout-min 90

## 5. Final training with the best configuration (estimate ≈ 10–20 min)
60 epochs with the Optuna configuration, W&B group `task1-final`. Checkpoints: `CKPT_DIR/task1/udae/last.pt` (every epoch) and `best.pt` (best validation score); the best model is also uploaded to W&B. Re-running this cell after a disconnect continues from the last finished epoch.

In [ ]:
!python -m src.task1.train --config $OUTPUT_DIR/task1/best_config.yaml --resume

## 6. Test-set evaluation (estimate ≈ 5 min)
All 36,690 test entries (every test image clean + 3 corruptions × 3 severities). Writes per-entry results, tables (CSV + LaTeX) per corruption type and severity, the comparison with the corrupted input and classical oracle baselines, and the figures (12 representative examples, failure cases, PSNR/SSIM vs severity, training curves) to `OUTPUT_DIR/task1/`.

In [ ]:
!python -m src.task1.evaluate

## 7. ONNX export for the application (estimate ≈ 1 min)
Exports `best.pt` to `ONNX_DIR/udae.onnx`, checks that ONNX Runtime matches PyTorch on 64 real test images (fails loudly otherwise) and writes the model card `udae.json`. Done before the ablation so the application has its model even if the session ends early.

In [ ]:
!python -m src.task1.export_onnx

## 8. Skip-connection ablation (estimate ≈ 10–17 min per run)
The main model has **no** skip connections (a genuine bottleneck). These three runs use the same configuration and epochs plus one limited skip connection at 16×16, 32×32 or 128×128, to measure what skips do: sharper outputs versus letting the corruption leak through. Each variant has its own folder (`CKPT_DIR/task1/udae_skip16`, ...), so the main model is never overwritten. Each cell resumes after a disconnect.

In [ ]:
!python -m src.task1.train --config $OUTPUT_DIR/task1/best_config.yaml --skip-resolutions 16 --resume

In [ ]:
!python -m src.task1.train --config $OUTPUT_DIR/task1/best_config.yaml --skip-resolutions 32 --resume

In [ ]:
!python -m src.task1.train --config $OUTPUT_DIR/task1/best_config.yaml --skip-resolutions 128 --resume

## 9. Evaluate the ablation variants and compare (estimate ≈ 2 min each)
Each variant is evaluated like the main model (outputs in `OUTPUT_DIR/task1/variants/<name>/`); the last cell builds the side-by-side table `OUTPUT_DIR/task1/tables/skip_ablation.{csv,tex}` and the figure `OUTPUT_DIR/task1/figures/skip_ablation_examples.png`.

In [ ]:
!python -m src.task1.evaluate --variant udae_skip16

In [ ]:
!python -m src.task1.evaluate --variant udae_skip32

In [ ]:
!python -m src.task1.evaluate --variant udae_skip128

In [ ]:
!python -m src.task1.evaluate --compare udae udae_skip16 udae_skip32 udae_skip128

## Where the results are (on Google Drive, `MyDrive/GenAI_A1/`)
- `checkpoints/task1/udae/best.pt` — final model; `checkpoints/task1/udae_skip*/` — ablation models
- `optuna/task1_udae.db` — the Optuna study; `outputs/task1/optuna/task1_udae/` — its report
- `outputs/task1/best_config.yaml` — selected hyperparameters
- `outputs/task1/eval/`, `outputs/task1/tables/`, `outputs/task1/figures/` — test results for the report
- `onnx/udae.onnx` + `onnx/udae.json` — model for the application